# 표 구조 점검의 변형 실험

정상 입력에서 조건 하나를 바꾸고 예상한 문제가 표시되는지 확인합니다.

In [4]:
from pathlib import Path

import pandas as pd

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
csv_path = project_root / "data" / "kodex200-price-fixture.csv"

prices = pd.read_csv(csv_path, parse_dates=["Date"])
print(f"불러오기 완료: {len(prices)}행 × {len(prices.columns)}열")
import io

def table_check(table):
    expected_columns = ["Date", "Open", "High", "Low", "Close", "Volume", "Change"]
    dates = table["Date"] if "Date" in table.columns else None
    date_type_ok = dates is not None and pd.api.types.is_datetime64_any_dtype(dates)
    dates_ready = date_type_ok and len(dates) > 0 and not dates.isna().any()
    missing_count = int(table.isna().sum().sum())
    duplicate_count = int(dates.duplicated().sum()) if dates is not None else None
    basic_index = isinstance(table.index, pd.RangeIndex) and table.index.equals(pd.RangeIndex(len(table)))
    date_range = "확인 불가"
    range_ok = None
    ascending = None
    if dates_ready:
        date_range = f"{dates.min().date()} ~ {dates.max().date()}"
        range_ok = dates.min() == pd.Timestamp("2024-01-02") and dates.max() == pd.Timestamp("2024-01-17")
        ascending = bool(dates.is_monotonic_increasing)

    checks = pd.DataFrame(
        [
            ("행 × 열", f"{len(table)} × {len(table.columns)}", len(table) == 12 and len(table.columns) == 7),
            ("열 이름", ", ".join(table.columns), list(table.columns) == expected_columns),
            ("인덱스", type(table.index).__name__, basic_index),
            ("Date 자료형", str(dates.dtype) if dates is not None else "열 없음", date_type_ok if dates is not None else None),
            ("전체 결측", f"{missing_count}개", missing_count == 0),
            ("중복 날짜", f"{duplicate_count}개" if duplicate_count is not None else "확인 불가", duplicate_count == 0 if duplicate_count is not None else None),
            ("날짜 정렬", "확인 불가" if ascending is None else ("오름차순" if ascending else "섞임"), ascending),
            ("날짜 범위", date_range, range_ok),
        ],
        columns=["확인 항목", "결과", "판정"],
    )
    checks["판정"] = checks["판정"].map(lambda value: "확인 불가" if value is None else ("통과" if value else "확인 필요"))
    display(checks)
    if (checks["판정"] != "통과").any():
        raise RuntimeError("표 구조 점검에서 확인할 항목이 있습니다. 위 점검표를 읽으세요.")
    print("표 구조 점검 통과")


def observe(table):
    try:
        table_check(table)
    except RuntimeError as error:
        print(error)

print("실험 준비 완료 — 각 사례는 정상 원본의 복사본으로 시작합니다.")


불러오기 완료: 12행 × 7열
실험 준비 완료 — 각 사례는 정상 원본의 복사본으로 시작합니다.


## E3-M · 가격 한 칸이 비었을 때

전체 결측은 1개, 날짜 범위는 그대로일까요?

In [5]:
text = csv_path.read_text().replace("32870,32962,", "32870,,")
observe(pd.read_csv(io.StringIO(text), parse_dates=["Date"]))

,확인 항목,결과,판정
0,행 × 열,12 × 7,통과
1,열 이름,"Date, Open, High, Low, Close, Volume, Change",통과
2,인덱스,RangeIndex,통과
3,Date 자료형,datetime64[ns],통과
4,전체 결측,1개,확인 필요
5,중복 날짜,0개,통과
6,날짜 정렬,오름차순,통과
7,날짜 범위,2024-01-02 ~ 2024-01-17,통과


표 구조 점검에서 확인할 항목이 있습니다. 위 점검표를 읽으세요.


## E3-D · 날짜가 겹쳐도 오름차순일 수 있다

중복과 정렬은 각각 어떻게 판정될까요?

In [6]:
case = prices.copy()
case.loc[5, "Date"] = case.loc[4, "Date"]
observe(case)

,확인 항목,결과,판정
0,행 × 열,12 × 7,통과
1,열 이름,"Date, Open, High, Low, Close, Volume, Change",통과
2,인덱스,RangeIndex,통과
3,Date 자료형,datetime64[ns],통과
4,전체 결측,0개,통과
5,중복 날짜,1개,확인 필요
6,날짜 정렬,오름차순,통과
7,날짜 범위,2024-01-02 ~ 2024-01-17,통과


표 구조 점검에서 확인할 항목이 있습니다. 위 점검표를 읽으세요.


## E3-S · 가운데 두 행의 순서만 바꾼다

날짜의 최소·최대가 같으면 정렬도 같을까요?

In [7]:
case = prices.iloc[[0, 1, 2, 3, 5, 4, 6, 7, 8, 9, 10, 11]].reset_index(drop=True)
observe(case)

,확인 항목,결과,판정
0,행 × 열,12 × 7,통과
1,열 이름,"Date, Open, High, Low, Close, Volume, Change",통과
2,인덱스,RangeIndex,통과
3,Date 자료형,datetime64[ns],통과
4,전체 결측,0개,통과
5,중복 날짜,0개,통과
6,날짜 정렬,섞임,확인 필요
7,날짜 범위,2024-01-02 ~ 2024-01-17,통과


표 구조 점검에서 확인할 항목이 있습니다. 위 점검표를 읽으세요.


## E3-T · 날짜를 문자열로 읽는다

날짜처럼 보여도 날짜 계산을 해도 될까요?

In [8]:
case = pd.read_csv(csv_path)
observe(case)

,확인 항목,결과,판정
0,행 × 열,12 × 7,통과
1,열 이름,"Date, Open, High, Low, Close, Volume, Change",통과
2,인덱스,RangeIndex,통과
3,Date 자료형,object,확인 필요
4,전체 결측,0개,통과
5,중복 날짜,0개,통과
6,날짜 정렬,확인 불가,확인 불가
7,날짜 범위,확인 불가,확인 불가


표 구조 점검에서 확인할 항목이 있습니다. 위 점검표를 읽으세요.


## E3-C · 날짜 값 하나가 깨져 있다

parse_dates 옵션만으로 모든 값이 날짜가 될까요?

In [9]:
text = csv_path.read_text().replace("2024-01-09", "not-a-date")
case = pd.read_csv(io.StringIO(text), parse_dates=["Date"])
observe(case)

,확인 항목,결과,판정
0,행 × 열,12 × 7,통과
1,열 이름,"Date, Open, High, Low, Close, Volume, Change",통과
2,인덱스,RangeIndex,통과
3,Date 자료형,object,확인 필요
4,전체 결측,0개,통과
5,중복 날짜,0개,통과
6,날짜 정렬,확인 불가,확인 불가
7,날짜 범위,확인 불가,확인 불가


표 구조 점검에서 확인할 항목이 있습니다. 위 점검표를 읽으세요.


## E3-N · 필요한 날짜 열이 없다

파일을 읽는 단계와 표를 검사하는 단계는 어디서 멈출까요?

In [10]:
text = csv_path.read_text().replace("Date,", "TradeDate,", 1)
try:
    pd.read_csv(io.StringIO(text), parse_dates=["Date"])
except ValueError as error:
    print("읽기 단계:", error)
observe(prices.rename(columns={"Date": "TradeDate"}))

읽기 단계: Missing column provided to 'parse_dates': 'Date'


,확인 항목,결과,판정
0,행 × 열,12 × 7,통과
1,열 이름,"TradeDate, Open, High, Low, Close, Volume, Change",확인 필요
2,인덱스,RangeIndex,통과
3,Date 자료형,열 없음,확인 불가
4,전체 결측,0개,통과
5,중복 날짜,확인 불가,확인 불가
6,날짜 정렬,확인 불가,확인 불가
7,날짜 범위,확인 불가,확인 불가


표 구조 점검에서 확인할 항목이 있습니다. 위 점검표를 읽으세요.


## E3-R · 첫 데이터 행을 뺀다

한 번의 조작이 몇 개의 점검 결과를 바꿀까요?

In [11]:
case = prices.iloc[1:].reset_index(drop=True)
observe(case)

,확인 항목,결과,판정
0,행 × 열,11 × 7,확인 필요
1,열 이름,"Date, Open, High, Low, Close, Volume, Change",통과
2,인덱스,RangeIndex,통과
3,Date 자료형,datetime64[ns],통과
4,전체 결측,0개,통과
5,중복 날짜,0개,통과
6,날짜 정렬,오름차순,통과
7,날짜 범위,2024-01-03 ~ 2024-01-17,확인 필요


표 구조 점검에서 확인할 항목이 있습니다. 위 점검표를 읽으세요.


## E3-I · 행 번호를 문자열 이름으로 바꾼다

이 파일의 기본 인덱스 조건과 다른 표의 타당성은 같은 판단일까요?

In [12]:
case = prices.copy()
case.index = [f"row-{i}" for i in range(len(case))]
observe(case)

,확인 항목,결과,판정
0,행 × 열,12 × 7,통과
1,열 이름,"Date, Open, High, Low, Close, Volume, Change",통과
2,인덱스,Index,확인 필요
3,Date 자료형,datetime64[ns],통과
4,전체 결측,0개,통과
5,중복 날짜,0개,통과
6,날짜 정렬,오름차순,통과
7,날짜 범위,2024-01-02 ~ 2024-01-17,통과


표 구조 점검에서 확인할 항목이 있습니다. 위 점검표를 읽으세요.
